In [2]:
%pip install sentence-transformers scikit-learn "psycopg[binary]" pgvector numpy

  Using cached sentence_transformers-6.3.0-py3-none-any.whl.metadata (20 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached numpy-2.5.5-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.15.0-cp314-cp314-win_amd64.whl.metadata (36 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.8.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.2-py3-none-any.whl.metadata (10.0 kB)
  Using cached httpcore2-2.16.2-py3-none-any.whl.metadata (28 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached truststore-0.10.5-py3-


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from sentence_transformers import SentenceTransformer

import numpy as np

c:\Users\USUARIO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
documentos =  [
    "El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático",
    "Solicitud de aumento de línea de crédito para cuenta empresarial",
    "El cliente no reconoce un cargo de $150 en su estado de cuenta",
    "Consulta sobre requisitos para abrir una cuenta de ahorros infantil",
    "Reclamo por doble cobro en transferencia interbancaria",
    "Solicitud de copia de estado de cuenta de los últimos 6 meses",
    "El cajero automático retuvo la tarjeta del cliente",
    "Consulta sobre tasa de interés para préstamo hipotecario",
    "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito",
    "Solicitud para cambiar el número de teléfono asociado a la cuenta",
    "El cliente reporta una transacción que no ha realizado con su tarjeta de crédito",
    "No está funcionando las transferencias ACH a las 08:00 am",
    "El usuario de banca en línea fue bloqueado y no puede entrar",
    "La app móvil se encuentra fuera de servicio",
    "El cliente solicita un préstam por consolidación de deudas",
    "La empresa ABC quiere cancelar su cuenta de cheques y abrir una nueva",
    "Se han incrementado las solicitudes de tarjeta de crédto para clientes con historial crediticio limitado",
    "Consulta para cancelar una tarjeta de crédito que ha sido robada",
    "El usuario de la app no puede ver el saldo de la tarjeta de crédito",
    "Solicitud para préstamo de pequeña empresa para capital de trabajo"
]

In [5]:
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embedding_docs = modelo.encode(documentos)

print(embedding_docs)        # vectores completos
print(embedding_docs.shape)  # (20, 384)
DIM = embedding_docs.shape[1]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1672.10it/s]


[[-0.00278847  0.09540856 -0.43310556 ... -0.14843068 -0.09072255
  -0.18465376]
 [ 0.08226307 -0.02862829 -0.31841314 ... -0.3605882  -0.2990758
  -0.06653975]
 [ 0.12315454  0.10069431 -0.08750795 ... -0.11724831 -0.15824169
  -0.2116466 ]
 ...
 [-0.1029171   0.2714748  -0.4644348  ... -0.18284103 -0.21163373
   0.11220942]
 [-0.19400737 -0.06343802 -0.26125324 ... -0.2359809  -0.17472564
  -0.17749882]
 [-0.08207162  0.00126006 -0.13292986 ... -0.15150867 -0.11735423
  -0.14535318]]
(20, 384)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
def buscar_documentos_similares(consulta, documentos, embedding_docs, modelo, top_k=3):

    # 1. Convierte la pregunta a embedding
    embedding_consulta = modelo.encode([consulta])

    # 2. Calcula la similitud entre todos los documentos y la consulta
    similitudes = cosine_similarity(embedding_consulta, embedding_docs)[0]

    # 3. Obtén los índices de los documentos más similares
    indices_top = np.argsort(similitudes)[::-1][:top_k]

    # 4. Devuelve los documentos más similares y sus puntuaciones de similitud
    resultados = [(documentos[i], similitudes[i]) for i in indices_top]

    return resultados

In [8]:
pregunta = "alguien uso mi tarjeta sin permiso"

resultados = buscar_documentos_similares(pregunta, documentos, embedding_docs, modelo, top_k=3)

for texto, score in resultados:
    print(f"Documento: {texto}\nSimilitud: {score:.4f}\n")

Documento: El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
Similitud: 0.6078

Documento: El cajero automático retuvo la tarjeta del cliente
Similitud: 0.6022

Documento: Consulta para cancelar una tarjeta de crédito que ha sido robada
Similitud: 0.5857



In [9]:
import time
import psycopg
from pgvector.psycopg import register_vector

# Ajusta usuario / contraseña / base según tu instalación
DSN = "host=localhost port=5432 dbname=postgres user=postgres password=postgres"

conn = psycopg.connect(DSN, autocommit=True)
conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
register_vector(conn)  # permite pasar arrays de numpy directamente

TABLA = "tickets_banco"

conn.execute(f"DROP TABLE IF EXISTS {TABLA}")
conn.execute(f"""
    CREATE TABLE {TABLA} (
        id        integer PRIMARY KEY,
        texto     text NOT NULL,
        categoria text,
        embedding vector({DIM})
    )
""")

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost database=postgres) at 0x29c28c728d0>

In [10]:
t0 = time.perf_counter()
with conn.cursor() as cur:
    cur.executemany(
        f"INSERT INTO {TABLA} (id, texto, embedding) VALUES (%s, %s, %s)",
        [(i, doc, emb) for i, (doc, emb) in enumerate(zip(documentos, embedding_docs))]
    )
print(f"Carga: {time.perf_counter() - t0:.3f} s")
print("Filas en pgvector:", conn.execute(f"SELECT count(*) FROM {TABLA}").fetchone()[0])

Carga: 0.054 s
Filas en pgvector: 20


In [11]:
def buscar_pgvector(consulta, top_k=3):
    vector = modelo.encode(consulta)
    filas = conn.execute(
        f"""
        SELECT texto, 1 - (embedding <=> %s::vector) AS similitud
        FROM {TABLA}
        ORDER BY embedding <=> %s::vector
        LIMIT %s
        """,
        (vector, vector, top_k),
    ).fetchall()
    return [(texto, float(score)) for texto, score in filas]

for texto, score in buscar_pgvector(pregunta):
    print(f"Documento: {texto}\nSimilitud: {score:.4f}\n")

Documento: El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
Similitud: 0.6078

Documento: El cajero automático retuvo la tarjeta del cliente
Similitud: 0.6022

Documento: Consulta para cancelar una tarjeta de crédito que ha sido robada
Similitud: 0.5857



In [12]:
preguntas_prueba = [
    ("alguien uso mi tarjeta sin permiso", "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito"),
    ("me quedé sin poder entrar a la banca en línea", "El usuario de banca en línea fue bloqueado y no puede entrar"),
    ("quiero un crédito para mi negocio", "Solicitud para préstamo de pequeña empresa para capital de trabajo"),
    ("el cajero se tragó mi tarjeta", "El cajero automático retuvo la tarjeta del cliente"),
    ("necesito mis estados de cuenta anteriores", "Solicitud de copia de estado de cuenta de los últimos 6 meses"),
]

In [13]:
def evaluar(buscar_fn, preguntas, top_k=3):
    aciertos = 0
    tiempos = []
    for pregunta, esperado in preguntas:
        t0 = time.perf_counter()
        resultados = buscar_fn(pregunta, top_k)
        tiempos.append(time.perf_counter() - t0)
        textos = [r[0] for r in resultados]
        if esperado in textos:
            aciertos += 1
    print(f"Aciertos en top-{top_k}: {aciertos}/{len(preguntas)}")
    print(f"Latencia promedio: {sum(tiempos)/len(tiempos)*1000:.1f} ms")

evaluar(buscar_pgvector, preguntas_prueba)

Aciertos en top-3: 4/5
Latencia promedio: 67.1 ms


In [14]:
vectores_q = [modelo.encode(p) for p, _ in preguntas_prueba]

SQL_TOP = f"SELECT id FROM {TABLA} ORDER BY embedding <=> %s::vector LIMIT 3"

tiempos = []
for _ in range(50):
    for v in vectores_q:
        t0 = time.perf_counter()
        conn.execute(SQL_TOP, (v,)).fetchall()
        tiempos.append((time.perf_counter() - t0) * 1000)

print(f"Latencia (solo pgvector): promedio {np.mean(tiempos):.1f} ms | p95 {np.percentile(tiempos, 95):.1f} ms")

Latencia (solo pgvector): promedio 1.8 ms | p95 2.8 ms


In [15]:
categorias = [
    "tarjetas", "prestamos", "fraude", "cuentas", "transferencias",
    "cuentas", "tarjetas", "prestamos", "fraude", "cuentas",
    "fraude", "canales_digitales", "canales_digitales", "canales_digitales", "prestamos",
    "cuentas", "tarjetas", "tarjetas", "canales_digitales", "prestamos",
]

with conn.cursor() as cur:
    cur.executemany(
        f"UPDATE {TABLA} SET categoria = %s WHERE id = %s",
        [(cat, i) for i, cat in enumerate(categorias)]
    )

conn.execute(f"CREATE INDEX IF NOT EXISTS idx_{TABLA}_categoria ON {TABLA} (categoria)")

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost database=postgres) at 0x29c28c72bd0>

In [16]:
v = modelo.encode("alguien uso mi tarjeta sin permiso")

SQL_BUSQ = f"""
    SELECT 1 - (embedding <=> %s::vector) AS similitud, categoria, texto
    FROM {TABLA}
    {{where}}
    ORDER BY embedding <=> %s::vector
    LIMIT 3
"""

print("SIN filtro:")
for score, cat, texto in conn.execute(SQL_BUSQ.format(where=""), (v, v)).fetchall():
    print(f"  {score:.3f} | {cat} | {texto}")

print("\nCON filtro categoria=fraude:")
for score, cat, texto in conn.execute(
        SQL_BUSQ.format(where="WHERE categoria = %s"), (v, "fraude", v)).fetchall():
    print(f"  {score:.3f} | {cat} | {texto}")

SIN filtro:
  0.608 | tarjetas | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
  0.602 | tarjetas | El cajero automático retuvo la tarjeta del cliente
  0.586 | tarjetas | Consulta para cancelar una tarjeta de crédito que ha sido robada

CON filtro categoria=fraude:
  0.576 | fraude | El cliente reporta una transacción que no ha realizado con su tarjeta de crédito
  0.491 | fraude | Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
  0.411 | fraude | El cliente no reconoce un cargo de $150 en su estado de cuenta


In [17]:
BENCH = "bench"
N = 20000

conn.execute(f"DROP TABLE IF EXISTS {BENCH}")
conn.execute(f"CREATE TABLE {BENCH} (id integer PRIMARY KEY, embedding vector({DIM}))")

rng = np.random.default_rng(42)
datos = rng.normal(size=(N, DIM)).astype("float32")

t0 = time.perf_counter()
with conn.cursor() as cur:
    for i in range(0, N, 1000):
        cur.executemany(
            f"INSERT INTO {BENCH} (id, embedding) VALUES (%s, %s)",
            [(j, datos[j]) for j in range(i, min(i + 1000, N))]
        )
print(f"Carga de {N} vectores: {time.perf_counter() - t0:.2f} s")

conn.execute(f"ANALYZE {BENCH}")

Carga de 20000 vectores: 4.23 s


<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost database=postgres) at 0x29c28c73290>

In [18]:
SQL_BENCH = f"SELECT id FROM {BENCH} ORDER BY embedding <=> %s::vector LIMIT 10"

def medir(n=100):
    tiempos = []
    for _ in range(n):
        q = rng.normal(size=DIM).astype("float32")
        t0 = time.perf_counter()
        conn.execute(SQL_BENCH, (q,)).fetchall()
        tiempos.append((time.perf_counter() - t0) * 1000)
    return np.mean(tiempos), np.percentile(tiempos, 95)

# Sin índice -> búsqueda exacta
prom, p95 = medir()
print(f"Latencia EXACTA (sin índice) con {N} vectores: promedio {prom:.1f} ms | p95 {p95:.1f} ms")

# Con índice HNSW -> búsqueda aproximada
t0 = time.perf_counter()
conn.execute(f"CREATE INDEX idx_{BENCH}_hnsw ON {BENCH} USING hnsw (embedding vector_cosine_ops)")
print(f"Construcción del índice HNSW: {time.perf_counter() - t0:.2f} s")

prom, p95 = medir()
print(f"Latencia HNSW con {N} vectores: promedio {prom:.1f} ms | p95 {p95:.1f} ms")

Latencia EXACTA (sin índice) con 20000 vectores: promedio 20.7 ms | p95 26.3 ms
Construcción del índice HNSW: 15.95 s
Latencia HNSW con 20000 vectores: promedio 4.7 ms | p95 6.4 ms


In [19]:
aciertos, total = 0, 0
for _ in range(50):
    q = rng.normal(size=DIM).astype("float32")

    # Aproximada (usa el índice HNSW)
    aprox = {r[0] for r in conn.execute(SQL_BENCH, (q,)).fetchall()}

    # Exacta (se desactiva el índice solo para esta consulta)
    conn.execute("SET enable_indexscan = off")
    exacta = {r[0] for r in conn.execute(SQL_BENCH, (q,)).fetchall()}
    conn.execute("SET enable_indexscan = on")

    aciertos += len(aprox & exacta)
    total += 10
print(f"Recall@10 (HNSW vs exacta): {aciertos/total:.2%}")

Recall@10 (HNSW vs exacta): 100.00%


In [20]:
for p_, esperado in preguntas_prueba:
    top = buscar_pgvector(p_, 3)
    ok = esperado in [t for t, _ in top]
    print("✅" if ok else "❌", p_)
    if not ok:
        print("   Esperado:", esperado)
        for t, s in top:
            print(f"   {s:.3f} | {t}")

❌ alguien uso mi tarjeta sin permiso
   Esperado: Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito
   0.608 | El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático
   0.602 | El cajero automático retuvo la tarjeta del cliente
   0.586 | Consulta para cancelar una tarjeta de crédito que ha sido robada
✅ me quedé sin poder entrar a la banca en línea
✅ quiero un crédito para mi negocio
✅ el cajero se tragó mi tarjeta
✅ necesito mis estados de cuenta anteriores
